In [1]:
import sys
!{sys.executable} -m pip install requests transformers wikipedia-api torch



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import requests
from transformers import pipeline
import wikipediaapi

# Initialize Wikipedia API with user-agent header
wiki_wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='AIResearchAssistant/1.0 (your.email@example.com)'
)

# Load summarization and QA pipelines
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0
Device set to use mps:0


In [3]:
def search_semantic_scholar(query, limit=10):
    url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={query}&limit={limit}&fields=title,abstract,url,authors,year"
    response = requests.get(url)
    print('Status code:', response.status_code)  # Debugging
    if response.status_code == 200:
        return response.json().get('data', [])
    else:
        print("Error fetching papers:", response.status_code)
        return []


In [4]:
def retrieve_and_summarize(query, limit=10):
    papers = search_semantic_scholar(query, limit)
    results = []
    for paper in papers:
        abstract = paper.get('abstract')
        if abstract:
            summary = summarizer(abstract, max_length=150, min_length=40, do_sample=False)[0]['summary_text']
        else:
            summary = "Abstract not available"
        results.append({
            'title': paper.get('title', 'No Title'),
            'summary': summary,
            'url': paper.get('url', 'URL not available'),
            'abstract': abstract if abstract else ""
        })
    return results


In [5]:
def answer_question(question, context):
    if not context:
        return "No context available for this paper."
    result = qa_pipeline(question=question, context=context)
    return result.get('answer', 'No answer found.')

import wikipediaapi

def get_wikipedia_summary_and_url(query):
    title = query.strip()
    page = wiki_wiki.page(title)
    if not page.exists():
        # fallback: search for closest match (simple heuristic)
        # since wikipediaapi does not have search, we implement a naive fallback:
        search_title = title.split()[-1].capitalize()  # last word fallback
        page = wiki_wiki.page(search_title)
    if not page.exists():
        return None, None
    return page.summary[:1000], page.fullurl


def combined_answer(question, paper_abstract):
    answer = answer_question(question, paper_abstract)
    if (
        answer.lower() in ["no answer found.", "no context available for this paper.", "", "n/a"]
        or len(answer) < 15
        or "www." in answer or ".org" in answer or ".com" in answer or ".net" in answer
    ):
        wiki_summary, wiki_url = get_wikipedia_summary_and_url(question)
        if wiki_summary:
            return f"Wikipedia Summary:\n{wiki_summary}\n\nRead more at: {wiki_url}"
        else:
            return "Sorry, no answer found in paper or Wikipedia."
    else:
        return f"Paper-based Answer:\n{answer}"


In [6]:
query = input("Enter your search query: ")
results = retrieve_and_summarize(query, limit=10)

if len(results) == 0:
    print("No papers found. Try another query or check your function/API.")
else:
    print(f"\nRetrieved {len(results)} papers for query: {query}\n")
    for i, res in enumerate(results):
        print(f"Paper {i+1}: {res['title']}")
        print(f"Summary: {res['summary']}")
        print(f"URL: {res['url']}\n")

    paper_index = int(input(f"Enter paper number (1-{len(results)}) to ask a question about: ")) - 1
    question = input("Enter your question about this paper or general topic: ")
    
    selected_paper_abstract = results[paper_index]['abstract']
    print("\nAnswer:")
    print(combined_answer(question, selected_paper_abstract))


Status code: 429
Error fetching papers: 429
No papers found. Try another query or check your function/API.


In [7]:
!python3 -m pip install SpeechRecognition pyaudio pyttsx3




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [8]:
!python3 -m pip install SpeechRecognition



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [9]:
!python3 -m pip install pyttsx3



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [10]:
import pyttsx3
print("pyttsx3 is installed and working!")


pyttsx3 is installed and working!


In [ ]:
import speech_recognition as sr
import pyttsx3
import os
from datetime import datetime
import wave

In [ ]:
# Enhanced Audio Input/Output Functions

def listen_query(timeout=10, phrase_time_limit=10):
    """
    Listen for audio input and convert to text using Google Speech Recognition
    
    Args:
        timeout: Time to wait for speech (seconds)
        phrase_time_limit: Max recording time per phrase (seconds)
    
    Returns:
        Recognized text or empty string if not recognized
    """
    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 4000  # Adjust for your mic sensitivity
    
    try:
        with sr.Microphone() as source:
            print("🎤 Listening... Please speak your query...")
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
        
        print("🔄 Processing audio...")
        query = recognizer.recognize_google(audio)
        print(f"✅ You said: {query}")
        return query
    
    except sr.UnknownValueError:
        print("❌ Sorry, I could not understand the audio. Please try again.")
        return ""
    except sr.RequestError as e:
        print(f"❌ Error accessing speech service: {e}")
        return ""
    except sr.Timeout:
        print("❌ No speech detected within the timeout period.")
        return ""
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return ""

def speak_text(text, rate=150, volume=0.9, voice_index=0):
    """
    Convert text to speech with customizable parameters
    
    Args:
        text: Text to speak
        rate: Speech rate (words per minute) - default 150
        volume: Volume level (0.0 to 1.0) - default 0.9
        voice_index: Voice selection (0=male, 1=female if available)
    """
    try:
        engine = pyttsx3.init()
        
        # Set speech rate
        engine.setProperty('rate', rate)
        
        # Set volume
        engine.setProperty('volume', volume)
        
        # Set voice
        voices = engine.getProperty('voices')
        if voice_index < len(voices):
            engine.setProperty('voice', voices[voice_index].id)
        
        print(f"🔊 Speaking... (Rate: {rate}, Volume: {volume})")
        engine.say(text)
        engine.runAndWait()
        print("✅ Speech completed.")
        
    except Exception as e:
        print(f"❌ Error during speech synthesis: {e}")

def save_audio_response(text, filename=None, rate=150, volume=0.9):
    """
    Save text-to-speech output to an audio file
    
    Args:
        text: Text to convert to speech
        filename: Output file name (auto-generated if None)
        rate: Speech rate
        volume: Volume level
    
    Returns:
        Path to saved audio file
    """
    try:
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"audio_response_{timestamp}.mp3"
        
        engine = pyttsx3.init()
        engine.setProperty('rate', rate)
        engine.setProperty('volume', volume)
        engine.save_to_file(text, filename)
        engine.runAndWait()
        
        print(f"✅ Audio saved to: {filename}")
        return filename
    
    except Exception as e:
        print(f"❌ Error saving audio: {e}")
        return None

def get_audio_settings():
    """Display and return current audio settings"""
    engine = pyttsx3.init()
    settings = {
        'rate': engine.getProperty('rate'),
        'volume': engine.getProperty('volume'),
        'voices': [v.name for v in engine.getProperty('voices')]
    }
    return settings


In [13]:
query = input("Enter your search query: ")


In [14]:
query = listen_query()


Please ask your research question (then wait)...
You said: tensorflow


In [18]:
query = listen_query()
results = retrieve_and_summarize(query, limit=5)

if len(results) == 0:
    print("No papers found. Try another query or check your function/API.")
else:
    for i, res in enumerate(results):
        print(f"Paper {i+1}: {res['title']}\nSummary: {res['summary']}\n")

    paper_index = int(input(f"Enter paper number (1-{len(results)}) to ask a question about: ")) - 1
    # Either listen for the question, or use input():
    question = listen_query()
    selected_paper_abstract = results[paper_index]['abstract']
    
    answer = combined_answer(question, selected_paper_abstract)
    print("\nAnswer:")
    print(answer)
    speak_text(answer)


Please ask your research question (then wait)...
You said: tensorflow
Status code: 200
Paper 1: TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems
Summary: TensorFlow is an interface for expressing machine learning algorithms, and an implementation for executing such algorithms. A computation expressed using TensorFlow can be executed with little or no change on a wide variety of heterogeneous systems. The system is flexible and can be used to express a wide range of algorithms.

Paper 2: This Paper Is Included in the Proceedings of the 12th Usenix Symposium on Operating Systems Design and Implementation (osdi '16). Tensorflow: a System for Large-scale Machine Learning Tensorflow: a System for Large-scale Machine Learning
Summary: Abstract not available

Paper 3: Hands-On Machine Learning with Scikit-Learn and TensorFlow: Concepts, Tools, and Techniques to Build Intelligent Systems
Summary: Abstract not available

Paper 4: GPT-Neo: Large Scale Autoregressive 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You said: what is tensorflow

Answer:
Wikipedia Summary:
TensorFlow is a software library for machine learning and artificial intelligence. It can be used across a range of tasks, but is used mainly for training and inference of neural networks. It is one of the most popular deep learning frameworks, alongside others such as PyTorch. It is free and open-source software released under the Apache License 2.0.
It was developed by the Google Brain team for Google's internal use in research and production. The initial version was released under the Apache License 2.0 in 2015. Google released an updated version, TensorFlow 2.0, in September 2019.
TensorFlow can be used in a wide variety of programming languages, including Python, JavaScript, C++, and Java, facilitating its use in a range of applications in many sectors.

Read more at: https://en.wikipedia.org/wiki/TensorFlow


In [ ]:
# Create requirements.txt file
requirements = """
transformers>=4.0.0
torch>=1.7.0
requests>=2.25.0
wikipedia-api>=0.5.4
SpeechRecognition>=3.8.1
pyttsx3>=2.90
google-cloud-speech>=2.10.0
pyaudio>=0.2.11
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

print("✅ requirements.txt file created with audio dependencies.")


requirements.txt file created.


In [ ]:
from IPython.display import FileLink
FileLink("requirements.txt")


/Users/ishmeetsingh/requirements.txt

In [ ]:
# ==========================================
# FULL AUDIO-INTEGRATED RESEARCH ASSISTANT
# ==========================================

print("🎯 AI Research Assistant with Audio I/O")
print("=" * 50)

# Show available audio settings
settings = get_audio_settings()
print(f"\n📋 Available Voices: {settings['voices']}")
print(f"📊 Current Speech Rate: {settings['rate']} wpm")
print(f"🔊 Current Volume: {settings['volume']}")

# Get audio input
print("\n" + "=" * 50)
speak_text("Welcome to the AI Research Assistant. Please ask your research question.", rate=130)
query = listen_query()

if query:
    speak_text(f"Searching for papers about {query}...", rate=140)
    results = retrieve_and_summarize(query, limit=5)
    
    if len(results) == 0:
        message = "No papers found. Please try another query."
        print(message)
        speak_text(message)
    else:
        # Display papers
        print(f"\n✅ Found {len(results)} papers:\n")
        for i, res in enumerate(results, 1):
            print(f"{i}. {res['title']}")
            print(f"   Summary: {res['summary']}\n")
        
        # Get paper selection via audio or input
        try:
            selection_msg = f"Which paper would you like to ask about? Say a number from 1 to {len(results)}."
            speak_text(selection_msg, rate=130)
            paper_num_input = listen_query()
            paper_index = int(paper_num_input) - 1
            
            if 0 <= paper_index < len(results):
                # Get question via audio
                speak_text("Please ask your question about this paper.", rate=130)
                question = listen_query()
                
                if question:
                    selected_paper_abstract = results[paper_index]['abstract']
                    answer = combined_answer(question, selected_paper_abstract)
                    
                    print("\n" + "=" * 50)
                    print("📖 ANSWER:")
                    print(answer)
                    print("=" * 50)
                    
                    # Speak the answer
                    speak_text("Here is the answer to your question.", rate=130)
                    speak_text(answer, rate=130)
                    
                    # Option to save audio response
                    save_audio = input("\n💾 Save audio response to file? (yes/no): ").lower()
                    if save_audio in ['yes', 'y']:
                        save_audio_response(answer)
                else:
                    speak_text("No question was understood.", rate=140)
            else:
                speak_text(f"Invalid selection. Please try again.", rate=140)
        
        except ValueError:
            speak_text("Please provide a valid number.", rate=140)
else:
    speak_text("No query was understood. Please try again.", rate=140)

print("\n✅ Session completed!")
